# 03 — Feature Engineering

This notebook creates a small set of candidate features from the cleaned employee dataset.
The goal is not to decide here that these features improve prediction. That will be tested later with cross-validation.

## 1. Imports

In [14]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TRAIN_CLEAN_PATH,
    PROCESSED_DATA_DIR,
)

pd.set_option("display.max_columns", 100)

## 2. Load Cleaned Dataset

In [15]:
train = pd.read_csv(TRAIN_CLEAN_PATH)

print("Shape:", train.shape)
display(train.head())

Shape: (1058, 32)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2


## 3. Create a working copy
The cleaned dataset remains unchanged. New candidate features are added to a copy.

In [16]:
features = train.copy()
original_columns = set(features.columns)

## 4. Career and Tenure Features
Several variables describe different parts of an employee's career. Ratios can add context that raw year counts do not capture by themselves.
For example, 5 years at a company means something different for someone with 6 total working years than for someone with 25.
If an employee has zero total working years or zero company tenure, the related ratio is set to 0 to avoid division by zero.

In [17]:
features["CareerStartAge"] = (
    features["Age"] - features["TotalWorkingYears"]
)

features["CompanyCareerRatio"] = np.where(
    features["TotalWorkingYears"] > 0,
    features["YearsAtCompany"] / features["TotalWorkingYears"],
    0,
)

features["CurrentRoleCompanyRatio"] = np.where(
    features["YearsAtCompany"] > 0,
    features["YearsInCurrentRole"] / features["YearsAtCompany"],
    0,
)

features["ManagerCompanyRatio"] = np.where(
    features["YearsAtCompany"] > 0,
    features["YearsWithCurrManager"] / features["YearsAtCompany"],
    0,
)

features["PromotionLagCompanyRatio"] = np.where(
    features["YearsAtCompany"] > 0,
    features["YearsSinceLastPromotion"] / features["YearsAtCompany"],
    0,
)

## 5. Satisfaction composite

The dataset contains several related satisfaction measures. A simple average gives us a broad satisfaction measure while keeping the original variables available.

In [18]:
satisfaction_columns = [
    "EnvironmentSatisfaction",
    "JobSatisfaction",
    "RelationshipSatisfaction",
]

features["SatisfactionAverage"] = (
    features[satisfaction_columns].mean(axis=1)
)

## 6. Income Features
Monthly income is positively skewed, so a log transformation may help linear models and neural networks.

`IncomePerJobLevel` adds context by relating income to organizational level.

In [19]:
features["LogMonthlyIncome"] = np.log1p(
    features["MonthlyIncome"]
)

features["IncomePerJobLevel"] = (
    features["MonthlyIncome"] / features["JobLevel"]
)

## 7. Overtime and Job Involvement Interaction

Overtime may have a different effect depending on job involvement. This creates a simple interaction while keeping both original variables.


In [20]:
overtime_flag = features["OverTime"].eq("Yes").astype(int)

features["OverTimeJobInvolvement"] = (
    overtime_flag * features["JobInvolvement"]
)

## 8. Review the Created Features

In [21]:
engineered_columns = [
    column for column in features.columns
    if column not in original_columns
]

print("Engineered features:", len(engineered_columns))
display(pd.DataFrame({"feature": engineered_columns}))

Engineered features: 9


,feature
0,CareerStartAge
1,CompanyCareerRatio
2,CurrentRoleCompanyRatio
3,ManagerCompanyRatio
4,PromotionLagCompanyRatio
5,SatisfactionAverage
6,LogMonthlyIncome
7,IncomePerJobLevel
8,OverTimeJobInvolvement


In [22]:
display(
    features[engineered_columns]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
CareerStartAge,1058.0,25.620038,6.974969,18.000000,20.000000,24.000000,30.000000,56.000000
CompanyCareerRatio,1058.0,0.675780,0.330437,0.000000,0.400000,0.800000,1.000000,1.000000
CurrentRoleCompanyRatio,1058.0,0.582271,0.332219,0.000000,0.360909,0.666667,0.845142,1.000000
ManagerCompanyRatio,1058.0,0.557299,0.334820,0.000000,0.333333,0.666667,0.800000,1.000000
PromotionLagCompanyRatio,1058.0,0.285711,0.338549,0.000000,0.000000,0.166667,0.500000,1.000000
SatisfactionAverage,1058.0,2.729679,0.632704,1.000000,2.333333,2.666667,3.333333,4.000000
LogMonthlyIncome,1058.0,8.563828,0.677062,6.917706,7.972897,8.497908,9.075379,9.903488
IncomePerJobLevel,1058.0,2981.261342,762.348887,1009.000000,2389.500000,2886.000000,3500.500000,4999.000000
OverTimeJobInvolvement,1058.0,0.791115,1.298728,0.000000,0.000000,0.000000,2.000000,4.000000


## 9. Sanity Checks

In [23]:
ratio_columns = [
    "CompanyCareerRatio",
    "CurrentRoleCompanyRatio",
    "ManagerCompanyRatio",
    "PromotionLagCompanyRatio",
]

print(
    "Career start age range:",
    features["CareerStartAge"].min(),
    "to",
    features["CareerStartAge"].max(),
)

for column in ratio_columns:
    print(
        column,
        "range:",
        round(features[column].min(), 3),
        "to",
        round(features[column].max(), 3),
    )

print(
    "Missing values created:",
    features[engineered_columns].isna().sum().sum(),
)

Career start age range: 18 to 56
CompanyCareerRatio range: 0.0 to 1.0
CurrentRoleCompanyRatio range: 0.0 to 1.0
ManagerCompanyRatio range: 0.0 to 1.0
PromotionLagCompanyRatio range: 0.0 to 1.0
Missing values created: 0


## 10. Feature Catalog for CSV File

In [24]:
feature_catalog = pd.DataFrame([
    ["CareerStartAge",
     "Approximate age when the employee began their working career"],
    ["CompanyCareerRatio",
     "Share of total working years spent at the current company"],
    ["CurrentRoleCompanyRatio",
     "Share of company tenure spent in the current role"],
    ["ManagerCompanyRatio",
     "Share of company tenure spent with the current manager"],
    ["PromotionLagCompanyRatio",
     "Years since last promotion relative to company tenure"],
    ["SatisfactionAverage",
     "Average environment, job, and relationship satisfaction"],
    ["LogMonthlyIncome",
     "Natural log transformation of monthly income"],
    ["IncomePerJobLevel",
     "Monthly income relative to job level"],
    ["OverTimeJobInvolvement",
     "Interaction between overtime status and job involvement"],
], columns=["feature", "description"])

display(feature_catalog)

,feature,description
0,CareerStartAge,Approximate age when the employee began their ...
1,CompanyCareerRatio,Share of total working years spent at the curr...
2,CurrentRoleCompanyRatio,Share of company tenure spent in the current role
3,ManagerCompanyRatio,Share of company tenure spent with the current...
4,PromotionLagCompanyRatio,Years since last promotion relative to company...
5,SatisfactionAverage,"Average environment, job, and relationship sat..."
6,LogMonthlyIncome,Natural log transformation of monthly income
7,IncomePerJobLevel,Monthly income relative to job level
8,OverTimeJobInvolvement,Interaction between overtime status and job in...


## 11. Save Engineered Features

In [25]:
FEATURED_DATA_PATH = PROCESSED_DATA_DIR / "train_featured.csv"

features.to_csv(FEATURED_DATA_PATH, index=False)

print("Saved to:", FEATURED_DATA_PATH)
print("Shape:", features.shape)

Saved to: C:\Users\jeffh\PycharmProjects\employee-attrition\data\processed\train_featured.csv
Shape: (1058, 41)
